# Random Molecular Dynamics and Langevin Equation

## Introduction
```{figure}https://upload.wikimedia.org/wikipedia/commons/c/c2/Brownian_motion_large.gif
:class: center
:width: 20%
From: https://upload.wikimedia.org/wikipedia/commons/c/c2/Brownian_motion_large.gif
```
### Brownian Motion

Microscopic particles suspended in a fluid exhibit erratic motion due to
collisions with surrounding molecules.

Examples:

- Pollen grains in water
- Dust particles in air
- Colloidal particles
- Molecular diffusion

A fully deterministic simulation of all molecules is often impossible,
so we construct an effective stochastic description.

### The Langevin Equation
In standard Molecular Dynamics (MD), we solve Newton's equations ($F = ma$). However, to simulate a system at a constant temperature $T$ or to model a particle in a solvent (like a protein in water), we need to include 
1. A friction force
2. A random fluctuating force

To do so, we use the **Langevin Equation**, 

$$m \frac{dv}{dt} = F_{ext}(x) - \gamma v + \sqrt{2 \gamma k_B T} \eta(t)$$

Where:
*   $F_{ext}(x)$: External forces (e.g., from a potential $U(x)$).
*   $-\gamma v$: **Friction/Drag force** (dissipation).
*   $\sqrt{2 \gamma k_B T} \eta(t)$: **Stochastic force** (fluctuation), where $\eta(t)$ is Gaussian white noise, which has the following properties:

\begin{align}
\langle \eta(t) \rangle &= 0,\\
\langle \eta(t) \eta(t') \rangle &= \delta(t-t').
\end{align}

The standard deviation is $\sigma^2 = 2 \gamma k_B T$. 

## Note on software
Although we are going to implement some of the simulations by ourselves, there are several MD packages worth visiting:
- [LAMMMPS](https://www.lammps.org/)
- [GROMACS](https://www.gromacs.org/)
- [OpenMM](https://openmm.org/)
- [DeepMD](https://docs.deepmodeling.com/projects/deepmd/en/stable/index.html#)
- ...
- See: <https://en.wikipedia.org/wiki/Category:Molecular_dynamics_software>


## Time integration
We cannot use the standard Verlet/Leapfrog algorithms in this cases, which are simplectic (they preserve phase-space volume) and time reversible, due to the random term ad the friction term in the Langevin equation, which introduce the following consequences:
1.  Friction makes the system non-Hamiltonian (energy is dissipated).
2.  The noise term is technically non-differentiable (Wiener process).

There are some new algorithms that can be implemented:

| Algorithm | Complexity | Stability | Accuracy ($x$) | Key Advantage | Disadvantage |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Euler-Maruyama** | Very Low | Poor | $O(\Delta t^{0.5})$ | Extremely simple to code. | Unstable; requires tiny time steps; poor energy/temperature control. |
| **Langevin Velocity Verlet** | Medium | Good | $O(\Delta t^{2})$ | Familiar to MD users; easy to implement. | Kinetic energy can be biased depending on how the friction is centered. |
| **G-JF (Grønbech-Jensen)** | Medium | **Excellent** | $O(\Delta t^{2})$ | **Exactly** reproduces Boltzmann distribution even for large $\Delta t$. | Slightly more algebraic overhead in the coefficients. |
| **BAOAB (Splitting)** | Medium | High | $O(\Delta t^{2})$ | Best for sampling configuration space (positions). | Velocity statistics are slightly less accurate than G-JF. |

---

### Deep Dive into the Integrators

### A. The Baseline: Euler-Maruyama
This is the stochastic version of the Forward Euler method:
\begin{align}
v_{n+1} &= v_n + \frac{F_n}{m}\Delta t - \frac{\gamma}{m} v_n \Delta t + \sqrt{\frac{2\gamma k_B T \Delta t}{m^2}} R_n,\\
x_{n+1} &= x_n + v_n \Delta t,
\end{align}
Where $R_n$ is a random number from a Normal distribution $N(0,1)$.

### B. The Standard: Modified Velocity Verlet
Most MD packages (like LAMMPS or GROMACS) use a variation of this. It splits the velocity update into two half-steps to account for the force, then adds the noise.
*   **Pros:** It reduces to the standard Verlet if $\gamma=0$ and $T=0$.
*   **Cons:** Because the friction depends on $v$, and $v$ is changing during the step, the "correct" velocity to use for the friction term is ambiguous.

See <https://www.sciencedirect.com/science/article/abs/pii/S0301010498002146>

### C. The Robust Choice: Grønbech-Jensen & Farago (G-JF)
This is currently the "gold standard" for Langevin simulations. It was designed so that for a harmonic oscillator, the calculated averages $\langle x^2 \rangle$ are **independent of the time step $\Delta t$**. 
It defines a discrete-time engine where the friction and noise are analytically integrated over the step.

See <https://arxiv.org/abs/1212.1244>


---

### Implementation and Comparison Code

Implement the euler-maruyama step. Use a external force as $-x$. Then run the code to check if the algorithms produce the same results.

In [ ]:
import numpy as np

# fix random seed for reproducibility
np.random.seed(1)

def euler_maruyama_step(x, v, f, dt, mass, gamma, kT):
    # Noise term
    noise = np.random.normal(0, 1)
    random_force = np.sqrt(2 * gamma * kT * dt) * noise
    
    # Update
    # YOUR CODE HERE
    pass
    
    return x_new, v_new, f_new

def gjf_step(x, v, f, dt, mass, gamma, kT):
    # 1. Calculate coefficients
    b = 1.0 / (1.0 + (gamma * dt / (2.0 * mass)))
    
    # 2. Generate random noise (Gaussian)
    # Variance is 2 * gamma * kT * dt
    noise = np.random.normal(0, np.sqrt(2 * gamma * kT * dt))
    
    # 3. Update Position
    x_new = x + b * dt * v + (b * dt**2 / (2 * mass)) * f + (b * dt / (2 * mass)) * noise
    
    # 4. We need the force at the NEW position to update velocity
    # For a simple harmonic oscillator: F = -k*x
    f_new = -1.0 * x_new  # Simplified: k=1
    
    # 5. Update Velocity (on-site velocity)
    v_new = b * v + (dt / (2 * mass)) * (b * f + f_new) + (b / mass) * noise
    
    return x_new, v_new, f_new

# Simulation Setup: High Time Step Test
dt = 0.5  # Large dt to test stability
n_steps = 1000
mass, gamma, kT = 1.0, 0.5, 1.0

# Initialize
x_em, v_em, f_em = 1.0, 0.0, -1.0
x_gjf, v_gjf, f_gjf = 1.0, 0.0, -1.0

traj_em = []
traj_gjf = []

for _ in range(n_steps):
    x_em, v_em, f_em = euler_maruyama_step(x_em, v_em, f_em, dt, mass, gamma, kT)
    x_gjf, v_gjf, f_gjf = gjf_step(x_gjf, v_gjf, f_gjf, dt, mass, gamma, kT)
    traj_em.append(x_em)
    traj_gjf.append(x_gjf)

In [ ]:
#%matplotlib widget
import matplotlib.pyplot as plt

# Plotting comparison
plt.figure(figsize=(12, 5))
plt.plot(traj_em, label='Euler-Maruyama', alpha=0.7)
plt.plot(traj_gjf, label='G-JF Integrator', alpha=0.7)
plt.axhline(0, color='black', lw=1)
plt.title(f"Integrator Stability Comparison (dt={dt})")
plt.ylabel("Position (x)")
plt.xlabel("Steps")
plt.legend()
plt.ylim(-10, 10) # EM often explodes
plt.savefig("fig/integrator_comp.png", dpi=90)
plt.show()


In [ ]:
import os
from IPython.display import Image, display
image_path="fig/integrator_comp.png"
# Display the image only if it exists
if os.path.exists(image_path):
    display(Image(filename=image_path))

#### Exercise: The "Explosion" Point

In Numerical Analysis, algorithms have a stability limit.

1. Gradually increase `dt` in the simulation above (try 0.1, 0.5, 1.0, 2.0).
2. What is the role of the random generator seed?
3. At what `dt` does the Euler-Maruyama algorithm "explode" (values go to infinity)?
4. Does the G-JF algorithm explode at the same point, or does it remain bounded?
5. Why is stability crucial when simulating complex molecules like proteins where forces can be very high?


## Calibration and testing 
It is important to always test the results. In this case, we can test the dynamics in several ways:

### Maxwell-Boltzmann distribution
The ultimate test of a Langevin integrator is if it reproduces the **Maxwell-Boltzmann distribution**.

In [ ]:
#%matplotlib widget
# Run a long G-JF simulation
steps = 50000
x, v, f = 0.0, 0.0, 0.0
v_list = []

for _ in range(steps):
    x, v, f = gjf_step(x, v, f, 0.1, 1.0, 0.5, 1.0)
    v_list.append(v)

# Plot Histogram vs Theory
plt.hist(v_list, bins=50, density=True, alpha=0.6, label='Simulated v')
v_range = np.linspace(min(v_list), max(v_list), 100)
# Theoretical Maxwell-Boltzmann in 1D: sqrt(m / 2*pi*kT) * exp(-mv^2 / 2kT)
theory = np.sqrt(1.0 / (2 * np.pi * 1.0)) * np.exp(-v_range**2 / (2 * 1.0))
plt.plot(v_range, theory, 'r--', label='Theoretical MB')
plt.title("Velocity Distribution Calibration")
plt.legend()
plt.show()


### The Equipartition Theorem
A key "calibration" test is to check if the simulation respects the **Equipartition Theorem**:  

$$\langle \frac{1}{2} m v^2 \rangle = \frac{1}{2} k_B T$$ (in 1D).

Calculate the average kinetic energy from your trajectory and compare it to $\frac{1}{2} kT$.

In [ ]:
kinetic_energies = 0.5 * mass * trajectory[:, 1]**2
avg_ke = np.mean(kinetic_energies)
expected_ke = 0.5 * kT

print(f"Measured <KE>: {avg_ke:.4f}")
print(f"Expected <KE>: {expected_ke:.4f}")
print(f"Error: {abs(avg_ke - expected_ke)/expected_ke * 100:.2f}%")


### Mean Squared Displacement (MSD)
How does the particle explore space? For a free particle ($F_{ext}=0$):
*   **Ballistic Regime** (Short times): $MSD(t) \sim t^2$
*   **Diffusive Regime** (Long times): $MSD(t) \sim 2Dt$


1.  Run the simulation for a **Free Particle** ($F_{ext} = 0$).
2.  Calculate the MSD: $MSD(\tau) = \langle (x(t+\tau) - x(t))^2 \rangle$.
3.  Plot it on a **Log-Log** scale.
Do this for both integration schemes.

In [ ]:
def calculate_msd(pos):
    n = len(pos)
    msd = np.zeros(n // 2)
    for tau in range(1, n // 2):
        diffs = pos[tau:] - pos[:-tau]
        msd[tau] = np.mean(diffs**2)
    return msd

# Free particle simulation (f=0)
x, v, f = 0.0, 0.0, 0.0
free_traj = []
for i in range(2000):
    x, v, _ = gjf_step(x, v, 0.0, 0.05, 1.0, 1.0, 1.0)
    free_traj.append(x)

msd = calculate_msd(np.array(free_traj))
plt.loglog(msd, label="MSD")
plt.loglog([1, 100], [0.01, 100], '--', label="Slope=1 (Diffusive)")
plt.loglog([1, 20], [0.001, 0.4], ':', label="Slope=2 (Ballistic)")
plt.legend()
plt.title("MSD: Ballistic to Diffusive Transition")
plt.show()


## Exercises 
### BBK Integrator (Brünger–Brooks–Karplus)
Implement and test the BBK integrator

\begin{align}
v_{n+1/2} &= \frac{(1 - \gamma\Delta t / 2)v_n + \frac{\Delta t}{2m}F_n + R_n}{1 + \gamma\Delta t / 2},\\
x_{n+1} &= x_n + v_{n+1/2}\Delta t .
\end{align}

### Energy "conservation"
Plot $E(t) = K + U$. What dou you see? is constant? How does it depends on $\gamma$?

### Underdamped vs Overdamped
The parameter $\gamma$ determines the "personality" of the dynamics.
*   **Low $\gamma$ (Underdamped):** The particle oscillates like a real oscillator but with occasional kicks.
*   **High $\gamma$ (Overdamped):** The particle looks like a random walk; its velocity resets almost instantly.

### Phase Space Exploration
Create a loop to simulate three different $\gamma$ values (0.01, 1.0, 10.0) and plot their **Phase Space** ($v$ vs $x$). 


### Effective temperature
Compare measured temperature:

$$
T_{eff}  = m \langle v^2 \rangle
$$

to true temperature, as a function of time.

### Sampling a Double-Well Potential
Langevin dynamics is often used to study **barrier crossing**.
Use the following potential: $$U(x) = a x^4 - b x^2$$

1.  Modify the `f_new` to the correct form.
2.  Set $a=1, b=2$.
3.  Observe the trajectory. Does the particle stay in one well or jump between them?
4.  How does increasing the temperature `kT` affect the jumping frequency?
5.  Look for the `Kramers transition`



### Time correlation Function
Compute 
$$
C_v(t) = \langle v(0) v(t) \rangle,
$$
and check if it decays exponentially, and the relationship with the exponential constant and the physicl parameters.

## Future topics
- MD Software: lammps, gromacs, OpenMM, ...
- Fokker-Planck equation
- Ornstein-Uhlenbeck process
- Andersen Thermostat